[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/02.%20Parte%202/15.%20Clase%2015/13Class%20NB.ipynb)

In [ ]:
!pip install -q yfinance pandas numpy matplotlib seaborn scipy scikit-learn statsmodels cvxpy pyomo

# Clase 11:	Algunas mejoras a los códigos para simular y optimizar portafolios 

[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres), 

*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)

+ Departamento de Matemáticas y Física
+ dsanchez@iteso.mx
+ Tel. 3669-34-34 Ext. 3069
+ Oficina: Cubículo 4, Edificio J, 2do piso

# 1. Motivación

En primer lugar, para poder bajar precios y información sobre opciones de Yahoo, es necesario cargar algunos paquetes de Python. En este caso, el paquete principal será Pandas. También, se usarán el Scipy y el Numpy para las matemáticas necesarias y, el Matplotlib y el Seaborn para hacer gráficos de las series de datos. Se usará el paquete **CVXPY** para optimización convexa con el enfoque de **Programación Convexa Disciplinada** (DCP), y **Pyomo** para programación estocástica (problema de dieta MILP).

Para instalar: `pip install cvxpy pyomo`

In [ ]:
#importar los paquetes que se van a usar
import pandas as pd
import numpy as np
import datetime
from datetime import datetime
import scipy.stats as stats
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.covariance as skcov
import cvxpy as cp
%matplotlib inline
pd.set_option('display.notebook_repr_html', True)
pd.set_option('display.max_columns', 6)
pd.set_option('display.max_rows', 10)
pd.set_option('display.width', 78)
pd.set_option('precision', 3)
#Funciones para portafolios
import portfolio_func
from pyomo.environ import *
infinity = float('inf')
import statsmodels.api as sm

In [2]:
assets = ['AAPL','MSFT','AA','AMZN','KO','QAI']
closes = portfolio_func.get_historical_closes(assets, '2016-01-01', '2017-09-22')

In [3]:
daily_returns=portfolio_func.calc_daily_returns(closes)
huber = sm.robust.scale.Huber()
#Mean and standar deviation returns
returns_av, scale = huber(daily_returns)

In [ ]:
# Modelo de portafolio con restricción de riesgo — CVXPY (DCP)
#
# maximizar   μ' w              (rendimiento esperado)
# sujeto a    w' Q w ≤ max_risk (restricción cuadrática de riesgo)
#             Σ wᵢ = 1          (presupuesto)
#             wᵢ ≥ 0            (sin ventas en corto)

# Datos de ejemplo (rendimientos anuales 1994-2013)
assets_list = ['Stocks', 'Bonds', 'Commodities']
T_years = list(range(1994, 2014))

# Rendimientos históricos simulados (ejemplo)
np.random.seed(42)
R_data = np.random.randn(len(T_years), len(assets_list)) * 0.05 + 0.03

mu_port = R_data.mean(axis=0)
Q_port = np.cov(R_data.T) * len(T_years)  # Suma de productos cruzados (como en Pyomo original)
max_risk = 0.00305

w_port = cp.Variable(len(assets_list))
objective = cp.Maximize(mu_port @ w_port)
constraints = [
    cp.quad_form(w_port, Q_port) <= max_risk,  # DCP: quad_form convexa ≤ constante
    cp.sum(w_port) == 1,                         # DCP: afín == afín
    w_port >= 0                                   # DCP: afín ≥ 0
]
prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.ECOS)

print(f"Status: {prob.status}")
print(f"Rendimiento esperado: {prob.value:.6f}")
for name, val in zip(assets_list, w_port.value):
    print(f"  {name}: {val:.4f}")

In [ ]:
!type dietdata.dat

In [ ]:
!pyomo solve --solver=glpk diet.py dietdata.dat

In [ ]:
!type results.yml

# 2. Uso de Pandas para descargar datos de precios de cierre

Una vez cargados los paquetes, es necesario definir los tickers de las acciones que se usarán, la fuente de descarga (Yahoo en este caso, pero también se puede desde Google) y las fechas de interés. Con esto, la función *DataReader* del paquete *pandas_datareader* bajará los precios solicitados.

**Nota**: Usualmente, las distribuciones de Python no cuentan, por defecto, con el paquete *pandas_datareader*. Por lo que será necesario instalarlo aparte. El siguiente comando instala el paquete en Anaconda:
*conda install -c conda-forge pandas-datareader *

# 3. Formulación del riesgo de un portafolio y simulación Montecarlo

In [ ]:
r=0.0001
results_frame = portfolio_func.sim_mont_portfolio(daily_returns,100000,r)

In [ ]:
#Sharpe Ratio
max_sharpe_port = results_frame.iloc[results_frame['Sharpe'].idxmax()]
#Menor SD
min_vol_port = results_frame.iloc[results_frame['SD'].idxmin()]

In [ ]:
plt.scatter(results_frame.SD,results_frame.Returns,c=results_frame.Sharpe,cmap='RdYlBu')
plt.xlabel('Volatility')
plt.ylabel('Returns')
plt.colorbar()
#Sharpe Ratio
plt.scatter(max_sharpe_port[1],max_sharpe_port[0],marker=(5,1,0),color='r',s=1000);
#Menor SD
plt.scatter(min_vol_port[1],min_vol_port[0],marker=(5,1,0),color='g',s=1000);

In [ ]:
pd.DataFrame(max_sharpe_port)

In [ ]:
pd.DataFrame(min_vol_port)

# 4. Optimización de portafolios

In [ ]:
N=5000
results_frame_optim = portfolio_func.optimal_portfolio(daily_returns,N,r)

In [ ]:
#Montecarlo
plt.scatter(results_frame.SD,results_frame.Returns,c=results_frame.Sharpe,cmap='RdYlBu')
plt.xlabel('Volatility')
plt.ylabel('Returns')
plt.colorbar()
#Markowitz
plt.plot(results_frame_optim.SD, results_frame_optim.Returns, 'b-o');

In [ ]:
#Sharpe Ratio
max_sharpe_port_optim = results_frame_optim.iloc[results_frame_optim['Sharpe'].idxmax()]
#Menor SD
min_vol_port_optim = results_frame_optim.iloc[results_frame_optim['SD'].idxmin()]

In [ ]:
#Markowitz
plt.scatter(results_frame_optim.SD,results_frame_optim.Returns,c=results_frame_optim.Sharpe,cmap='RdYlBu');
plt.xlabel('Volatility')
plt.ylabel('Returns')
plt.colorbar()
#Sharpe Ratio
plt.scatter(max_sharpe_port_optim[1],max_sharpe_port_optim[0],marker=(5,1,0),color='r',s=1000);
#SD
plt.scatter(min_vol_port_optim[1],min_vol_port_optim[0],marker=(5,1,0),color='g',s=1000);

In [ ]:
pd.DataFrame(max_sharpe_port_optim)

In [ ]:
pd.DataFrame(min_vol_port_optim)